# Joint, marginal and conditional probability
All three read from one contingency table of the Titanic data, plus conditional probability by counting.

In [ ]:
import itertools
from fractions import Fraction
import pandas as pd
import plotly.express as px

df = pd.read_csv('data/titanic_train.csv')
# X = passenger class (1, 2, 3), Y = survived (0 = died, 1 = survived)
counts = pd.crosstab(df['Pclass'], df['Survived'])
counts

## 1. Joint probabilities
`normalize='all'` divides every cell by the 891 passengers.

In [ ]:
joint = pd.crosstab(df['Pclass'], df['Survived'], normalize='all')
print(joint.round(4))
# a joint distribution covers every combination, so it sums to 1
print('sum of all cells:', joint.values.sum())

## 2. Marginal probabilities
`margins=True` adds the row and column totals.

In [ ]:
both = pd.crosstab(df['Pclass'], df['Survived'], normalize='all', margins=True)
both.round(4)

In [ ]:
# a marginal is a row or column sum of the joint table
print('P(Y = 0) =', round(joint[0].sum(), 4))
print('P(X = 3) =', round(joint.loc[3].sum(), 4))

## 3. Conditional probabilities
`normalize='index'` divides each row by its own total: P(Y | X). `normalize='columns'` divides each column: P(X | Y).

In [ ]:
# each row sums to 1: P(survived | class)
pd.crosstab(df['Pclass'], df['Survived'], normalize='index').round(4)

In [ ]:
# each column sums to 1: P(class | survived)
pd.crosstab(df['Pclass'], df['Survived'], normalize='columns').round(4)

In [ ]:
# the same number from the formula P(A | B) = P(A and B) / P(B)
p_joint = joint.loc[3, 0]            # P(died and class 3)
p_class3 = joint.loc[3].sum()        # P(class 3)
print('P(died | class 3) =', round(p_joint / p_class3, 4))
print('P(died) =', round(joint[0].sum(), 4))

In [ ]:
cond = pd.crosstab(df['Pclass'], df['Survived'], normalize='index')
px.bar(x=cond.index.astype(str), y=cond[0], labels={'x': 'class', 'y': 'P(died | class)'})

## 4. Conditional probability by counting
Reduce the sample space to B, then count A inside it.

In [ ]:
def conditional(space, A, B):
    reduced = [o for o in space if B(o)]          # outcomes where B happened
    hits = [o for o in reduced if A(o)]           # of those, where A also happened
    return Fraction(len(hits), len(reduced)), len(hits), len(reduced)

coins = list(itertools.product('HT', repeat=3))
print('3 coins:', conditional(coins, lambda o: o.count('H') >= 2, lambda o: o.count('H') >= 1))

dice = list(itertools.product(range(1, 7), repeat=2))
print('sum 7 | die 1 odd:', conditional(dice, lambda o: sum(o) == 7, lambda o: o[0] % 2 == 1))
print('die 1 = 2 | sum <= 5:', conditional(dice, lambda o: o[0] == 2, lambda o: sum(o) <= 5))

## 5. Bayes' theorem on five passengers

In [ ]:
toy = pd.DataFrame({'sex': ['male', 'male', 'female', 'male', 'female'],
                    'died': [1, 1, 1, 0, 0]})
p_male = (toy['sex'] == 'male').mean()
for d in [1, 0]:
    group = toy[toy['died'] == d]
    prior = len(group) / len(toy)                     # P(died = d)
    likelihood = (group['sex'] == 'male').mean()      # P(male | died = d)
    print('died =', d, ': P(died = d | male) =', round(likelihood * prior / p_male, 4))